# 07 — Options and termo

**The question:** *What does PETR4's front-expiry option chain look like, how
did one series trade, and which contracts were actually exercised?*

B3's COTAHIST tape carries options and forwards alongside the cash market, and
this API keeps them as **separate shapes** on purpose — because they are
different objects with different return semantics, and two of them have no
return semantics at all.

| `tpmerc` | What it is | Endpoint |
| --- | --- | --- |
| 070 / 080 | option quotes (call / put) | `option_chain`, `option_history` |
| 012 / 013 | option **exercise events** | `option_exercises` |
| 017 | **auction prints** | the `auctions` view |
| 030 | termo (forward) | `termo_history` |

Exercises and auctions are **events, not quotes**. Never compute a return over
either.

In [ ]:
# The SDK is not on PyPI. From the repository root:
#
#     pip install -e sdk/
#
# Auth is the shared publishable key printed in the docs. It is for TESTING:
# everyone reading the docs has the same one, so it identifies the project and
# not you. It puts you on the ANONYMOUS tier. Set SILO_TOKEN to a GitHub
# sign-in token (notebook 00) to run signed in.
import os

os.environ.setdefault("SILO_URL", "https://zcjbtpxuhdekpwcxmepn.supabase.co")
os.environ.setdefault(
    "SILO_ANON_KEY", "sb_publishable__yfFQsykAglrvc9GS6_PYw_B24ex437"
)

import pandas as pd

from silo_client import SiloClient

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

silo = SiloClient()
print(f"tier            : {silo.tier}")
print(f"catalog version : {silo.catalog()['version']}")

In [ ]:
def as_of(*datasets: str) -> pd.DataFrame:
    """Print how fresh every dataset this notebook relies on actually is.

    Run this FIRST, every time. A stale warehouse then shows up in the output
    instead of being silently baked into a number further down.

      as_of            the newest period that has landed AND has elapsed.
                       This is freshness.
      complete_through the newest period classified COMPLETE — what the
                       default windows serve.
      newest_period    the newest period KEY. It can sit in the FUTURE when a
                       family files forward-dated (FIP is keyed 31-December).
                       Never read this as freshness.
      landed_at        when ingest last SUCCEEDED. A later failed run never
                       advances it.
      notes            a caveat the dates cannot carry. Printed in full below,
                       never summarised, never dropped.
    """
    cov = pd.DataFrame(silo.coverage())
    rows = cov[cov["dataset"].isin(datasets)].copy()
    missing = set(datasets) - set(rows["dataset"])
    if missing:
        raise RuntimeError(f"coverage() has no row for {sorted(missing)}")
    print(
        rows[
            ["dataset", "as_of", "complete_through", "newest_period", "landed_at"]
        ].to_string(index=False)
    )
    for r in rows.itertuples():
        if r.notes:
            print(f"\n  CAVEAT [{r.dataset}]\n  {r.notes}")
    return rows.set_index("dataset")

In [ ]:
COVERAGE = as_of("derivatives", "quotes")

## A chain needs a prefix, and the server insists

An unfiltered whole-market chain is the slowest query on the API, so
`option_chain` **requires** a codneg prefix of at least three characters. The
SDK checks locally for `option_exercises`; the server enforces both.

In [ ]:
from silo_client.client import SiloError

try:
    silo.option_chain("PE")
except SiloError as exc:
    print("refused:", exc.body)

In [ ]:
PREFIX = "PETR"

chain = pd.DataFrame(silo.option_chain(PREFIX, limit=200))
chain["expiry"] = pd.to_datetime(chain["expiry"])

session = chain["trade_date"].iloc[0]
print(f"{len(chain)} rows, session {session} "
      f"(derivatives as_of {COVERAGE.loc['derivatives', 'as_of']})")
print(f"anonymous option_chain ceiling: "
      f"{silo.limits()['tiers']['anon']['option_chain_rows']} rows "
      f"(the default is 100; a larger p_limit is CLAMPED, not refused)")
print()
chain.head()

### `underlying_ticker` is published, not parsed from the code

This is the column that makes the chain usable. COTAHIST puts the
**underlying's** ISIN on an option row, and that ISIN is joined here to the same
session's cash print.

The codneg root cannot do this: `PETR`-prefixed options split between `PETR3`
and `PETR4`, and only the ISIN says which. It is `null` when the underlying had
no cash print that day — never guessed.

In [ ]:
print(chain.groupby(["underlying_ticker", "side"], dropna=False)
      .size().to_frame("rows").to_string())
print()
null_underlying = chain["underlying_ticker"].isna().sum()
print(f"rows with a NULL underlying_ticker: {null_underlying}")
print("(the underlying had no cash print that session — not a lookup failure)")

### The front expiry

In [ ]:
front = chain["expiry"].min()
fe = chain[chain["expiry"] == front].sort_values(["side", "strike"])

print(f"front expiry: {front:%Y-%m-%d}   ({len(fe)} series traded on {session})\n")
print(fe[["codneg", "side", "strike", "close", "trades", "quantity",
          "volume", "underlying_ticker"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

spot_rows = fe["underlying_ticker"].dropna()
spot = None
if len(spot_rows):
    spot = silo.quote_latest(spot_rows.iloc[0])[0]["close"]

fig, ax = plt.subplots(figsize=(10, 4))
for side, grp in fe.groupby("side"):
    ax.plot(grp["strike"], grp["close"], marker="o", markersize=3, label=side)
if spot is not None:
    ax.axvline(spot, color="grey", linestyle="--", linewidth=1,
               label=f"{spot_rows.iloc[0]} close {spot:,.2f}")
ax.set_xlabel("strike (BRL)")
ax.set_ylabel("option close (BRL)")
ax.set_title(f"{PREFIX} chain, expiry {front:%Y-%m-%d}, session {session}")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print("Only strikes that TRADED that session appear. A strike with no print is")
print("absent, not zero — there is no synthetic quote anywhere in this chart.")

## One series' history

`option_history` is `quote_history`'s shape plus `side`, `strike`, `expiry` and
`underlying_ticker`. It has **no `p_after` cursor** — deliberately: an option
series is short-lived, so a window that exceeds one page is a mistake rather
than a walk. It refuses and asks you to narrow.

In [ ]:
busiest = fe.sort_values("trades", ascending=False).iloc[0]
CODNEG = busiest["codneg"]
print(f"{CODNEG}: {busiest['side']} strike {busiest['strike']:,.2f} "
      f"expiry {busiest['expiry']:%Y-%m-%d}\n")

hist = pd.DataFrame(silo.option_history(CODNEG, start="2026-01-01"))
if hist.empty:
    print("no prints in this window")
else:
    hist["trade_date"] = pd.to_datetime(hist["trade_date"])
    hist = hist.set_index("trade_date").sort_index()
    print(f"{len(hist)} sessions, {hist.index.min().date()} .. "
          f"{hist.index.max().date()}")
    print(hist[["close", "trades", "quantity", "volume", "adjusted"]].tail(10)
          .to_string(float_format=lambda v: f"{v:,.2f}"))

Note `adjusted: false` here too. Nothing in this API is corporate-action
adjusted, options included — and an option series spanning a split is a series
whose strike and quote mean different things on either side of it.

## Exercises are events, not a price series

`option_exercises` serves `tpmerc` 012 (call) and 013 (put): **one print per
exercise**, with an `exercise_price` and no return semantics whatsoever. They do
not belong on a chart with closes, and a "return" between two of them is not a
number.

In [ ]:
ex = pd.DataFrame(silo.option_exercises(PREFIX, start="2026-06-01", limit=500))
print(f"{len(ex)} exercise events for {PREFIX} since 2026-06-01")
print(f"anonymous ceiling: "
      f"{silo.limits()['tiers']['anon']['option_exercises_rows']} rows "
      f"(clamped, not refused)\n")
ex[["codneg", "trade_date", "side", "strike", "exercise_price",
    "trades", "quantity", "volume", "underlying_ticker"]].head(10)

In [ ]:
if not ex.empty:
    by_day = (ex.assign(trade_date=pd.to_datetime(ex["trade_date"]))
              .groupby(["trade_date", "side"])
              .agg(events=("codneg", "size"),
                   quantity=("quantity", "sum"),
                   volume=("volume", "sum"))
              .tail(10))
    print("exercise activity by session — a COUNT and a SUM, never a return:\n")
    print(by_day.to_string(float_format=lambda v: f"{v:,.0f}"))

Auction prints (`tpmerc` 017) live on the `auctions` **view** and carry the same
warning. Being a view, it pages with `limit`/`offset` and **truncates** rather
than refusing — read `Content-Range`, or let the SDK raise.

Two practical notes. Auction prints are **sparse** — whole weeks pass without
one — so an empty result means "no auction in that window", not a failed query.
And an unbounded sort over the whole tape will not finish inside the anonymous
3-second budget: bound the window.

In [ ]:
WINDOW = "(trade_date.gte.2026-07-01,trade_date.lte.2026-09-16)"

auc = pd.DataFrame(silo.view("auctions", order="trade_date.desc",
                             limit=10, **{"and": WINDOW}))
if auc.empty:
    print("no auction prints in this window — sparse by nature, not an error")
else:
    print(f"{len(auc)} auction prints\n")
    print(auc[["ticker", "trade_date", "board", "close", "trades",
               "quantity", "volume"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    print()
    print("One print per auction. No open/close sequence, so no return.")

### And here is `57014`, live

Notebook 00 described the statement timeout without provoking one. The
`auctions` view provokes it reliably at the anonymous tier if you ask it to sort
the whole tape — the budget is **3 seconds** (8 signed in), and a query that
exceeds it is cancelled and returns **nothing**, never a partial result.

In [ ]:
from silo_client.client import SiloTimeout

try:
    silo.view("auctions", order="trade_date.desc", limit=5)   # unbounded window
except SiloTimeout as exc:
    print(f"{type(exc).__name__}: HTTP {exc.status}")
    print(" ", exc.body)
    print()
    print("Not an outage, and not a partial answer. Bound the window and retry.")
else:
    print("it finished this time — the database was warm.")
    print("Cold, the same call exceeds the 3s anonymous budget. Bound the")
    print("window anyway rather than relying on the cache being hot.")

## Termo: the grain has a third part

A forward's identity is `(codneg, trade_date, **term_days**)`. The same code on
the same session can print at several terms, and those are different contracts.

Termo codnegs carry a `T` suffix — `PETR4T`, not `PETR4`.

In [ ]:
TERMO = "PETR4T"

t = pd.DataFrame(silo.termo_history(TERMO, start="2026-06-01"))
print(f"{len(t)} rows for {TERMO}\n")
t[["codneg", "trade_date", "term_days", "close", "trades", "quantity",
   "volume"]].tail(10)

In [ ]:
if not t.empty:
    dup = t.groupby("trade_date")["term_days"].nunique()
    multi = dup[dup > 1]
    print(f"sessions with more than one term printing: {len(multi)} of {len(dup)}")
    if len(multi):
        day = multi.index[-1]
        print(f"\n{day}:")
        print(t[t["trade_date"] == day][["term_days", "close", "trades",
                                         "quantity", "volume"]]
              .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
        print()
        print("Pivoting this on trade_date alone would collapse distinct")
        print("contracts. term_days is part of the key, not a column to drop.")

## Where this goes next

* Notebook `01` is the cash tape these derive from — same unadjusted prices,
  same paging rules, but with a cursor.
* Notebook `08` is the other derivative-adjacent surface: securities lending and
  short interest.

---

## The rules this notebook obeyed

* **Nothing was filled.** No forward-fill, no interpolation, no carried-forward
  last observation. A gap in a chart is a gap in the filings.
* **Every caveat was printed beside its number** — `coverage().notes`,
  `catalog().regime_breaks`, `catalog().applicability`, `float_basis` — rather
  than left in a docstring somewhere.
* **Freshness came from `coverage()`**, called before anything was claimed.

The contract these rules come from is
[Conventions & limits](https://octo-98895abd.mintlify.site/api-docs/conventions),
and its machine-readable twin is `POST /rpc/catalog`.